# Fantasy Football Team Selection Optimization

## Problem Statement

I love watching football and enjoy participating in fantasy football leagues. For this season, I want to build the best possible team within a fixed budget and according to specific rules. My goal is to select a squad of 15 players from the Premier League that maximizes the total points they scored last season, while respecting budget, squad composition, and club player limits.

## Given Rules

1. **Budget:**

   * I have £100 million to spend on my entire squad.

2. **Squad Size & Composition:**

   * Exactly 15 players must be selected, structured as follows:

     * 2 Goalkeepers (GK)
     * 5 Defenders (DEF)
     * 5 Midfielders (MID)
     * 3 Forwards (FWD)

3. **Player Club Limit:**

   * I can select a maximum of 3 players from the same Premier League club.

## Problem Solving Approach

### Step 1: Define the decision variables

For each player $i$, define a binary variable $x_i$:

$$
x_i = \begin{cases}
1 & \text{if player } i \text{ is selected} \\
0 & \text{otherwise}
\end{cases}
$$

### Step 2: Collect the input data for each player

Each player $i$ has the following attributes:

* Position $pos_i$ — one of GK, DEF, MID, or FWD
* Price $p_i$ — cost in £ millions
* Total points $s_i$ — points scored last year
* Club $club_i$ — Premier League club name

### Step 3: Formulate the optimization model

**Objective:**

Maximize the total points of the selected players:

$$
\max \sum_{i} s_i \, x_i
$$

**Constraints:**

1. **Squad Size:**

Select exactly 15 players:

$$
\sum_i x_i = 15
$$

2. **Position Requirements:**

Exactly 2 Goalkeepers:

$$
\sum_{i : pos_i = \text{GK}} x_i = 2
$$

Exactly 5 Defenders:

$$
\sum_{i : pos_i = \text{DEF}} x_i = 5
$$

Exactly 5 Midfielders:

$$
\sum_{i : pos_i = \text{MID}} x_i = 5
$$

Exactly 3 Forwards:

$$
\sum_{i : pos_i = \text{FWD}} x_i = 3
$$

3. **Budget Constraint:**

Total cost cannot exceed £100 million:

$$
\sum_i p_i \, x_i \leq 100
$$

4. **Club Player Limit:**

For each club $c$, select at most 3 players:

$$
\sum_{i : club_i = c} x_i \leq 3
$$

5. **Binary selection variables:**

$$
x_i \in \{0, 1\} \quad \forall i
$$

## Data Format Example

| player\_name | position | price | points | club           |
| ------------ | -------- | ----- | ------ | -------------- |
| John Smith   | GK       | 5.0   | 120    | Manchester Utd |
| Alex Johnson | DEF      | 6.5   | 110    | Chelsea        |
| Mike Brown   | MID      | 8.0   | 140    | Liverpool      |
| David Green  | FWD      | 9.5   | 150    | Arsenal        |
| ...          | ...      | ...   | ...    | ...            |

* **price** is in £ millions.
* **points** is total points scored last season.
* Positions are one of GK, DEF, MID, FWD.
* Club is the Premier League club name.


By solving this integer linear optimization problem, I can determine the combination of players that maximizes total fantasy points under my budget and team rules, giving me the best possible squad to enjoy the football season!



In [1]:
%pip install gurobipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.5/14.5 MB 30.1 MB/s eta 0:00:00


In [2]:
import pandas as pd
from itertools import product

import gurobipy as gp
from gurobipy import GRB

In [16]:
import pandas as pd
from google.colab import files

# Upload the file
uploaded = files.upload()

# Get the filename
file_name = "Fantasy_Premier_League_Players.csv"

# Read the CSV into a pandas DataFrame
import pandas as pd
df = pd.read_csv(file_name)

# Display the first few rows of the DataFrame
print(df.head())

Saving Fantasy_Premier_League_Players.csv to Fantasy_Premier_League_Players (2).csv
     Player           Team Position    Price  Total Points
0     Isak      Newcastle      FWD   £10.5m            211
1     Wood   Nottm Forest      FWD    £7.5m            200
2    Bowen       West Ham      FWD    £8.0m            193
3  Watkins    Aston Villa      FWD    £9.0m            186
4    Wissa      Brentford      FWD    £7.5m            185


In [17]:
df = df.set_index('Player')
df


,Team,Position,Price,Total Points
Player,,,,
Isak,Newcastle,FWD,£10.5m,211
Wood,Nottm Forest,FWD,£7.5m,200
Bowen,West Ham,FWD,£8.0m,193
Watkins,Aston Villa,FWD,£9.0m,186
Wissa,Brentford,FWD,£7.5m,185
...,...,...,...,...
Bayindir,Man Utd,GKP,£5.0m,8
Neto,Bournemouth,GKP,£4.5m,7
Valdimarsson,Brentford,GKP,£4.0m,2


In [40]:
# Fix team name inconsistency
df['Team'] = df['Team'].replace({"Nott'm Forest": "Nottm Forest"})
df['Position'] = df['Position'].str.strip()
df['Team'] = df['Team'].str.strip()

# Clean Price column ('£10.5m' → 10.5 float)
df['Price'] = df['Price'].replace('[£m]', '', regex=True).astype(float)

# Make sure Player is a column, not only index
if df.index.name == 'Player':
    df = df.reset_index()

# Model setup
m = gp.Model("FantasyTeam")
ids = list(range(len(df)))

# Set the variables
x = m.addVars(ids, vtype=GRB.BINARY, name="pick")


# Objective: maximize total points
m.setObjective(
    gp.quicksum(df.iloc[i]['Total Points'] * x[i] for i in ids),
    GRB.MAXIMIZE
)

# Constraints
m.addConstr(gp.quicksum(x[i] for i in ids) == 15, "TotalPlayers")
m.addConstr(gp.quicksum(x[i] for i in ids if df.iloc[i]['Position'] == 'GKP') == 2, "GKP")
m.addConstr(gp.quicksum(x[i] for i in ids if df.iloc[i]['Position'] == 'DEF') == 5, "DEF")
m.addConstr(gp.quicksum(x[i] for i in ids if df.iloc[i]['Position'] == 'MID') == 5, "MID")
m.addConstr(gp.quicksum(x[i] for i in ids if df.iloc[i]['Position'] == 'FWD') == 3, "FWD")
m.addConstr(gp.quicksum(df.iloc[i]['Price'] * x[i] for i in ids) <= 100, "Budget")

for club in df['Team'].unique():
    m.addConstr(gp.quicksum(x[i] for i in ids if df.iloc[i]['Team'] == club) <= 3, f"Team_{club}")

# Solve and check feasibility

m.optimize()

if m.status == GRB.INFEASIBLE:
    print("Model is infeasible! Computing IIS to diagnose...")
    m.computeIIS()
    print("\nIrreducible Inconsistent Subsystem (IIS) constraints:")
    for c in m.getConstrs():
        if c.IISConstr:
            print(f" - {c.ConstrName}")
else:
    print("\nOptimal solution found.\n")
    team_rows = [i for i in ids if x[i].X > 0.5]
    team = df.iloc[team_rows].copy()
    print(team[['Player', 'Position', 'Price', 'Total Points', 'Team']].to_string(index=False))
    print("\nTotal Price:", round(team['Price'].sum(), 1))
    print("Total Points:", int(team['Total Points'].sum()))


Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (linux64 - "Ubuntu 22.04.4 LTS")

CPU model: Intel(R) Xeon(R) CPU @ 2.20GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 1 physical cores, 2 logical processors, using up to 2 threads

Optimize a model with 26 rows, 196 columns and 784 nonzeros
Model fingerprint: 0xf74f58de
Variable types: 0 continuous, 196 integer (196 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+01]
  Objective range  [1e+00, 3e+02]
  Bounds range     [1e+00, 1e+00]
  RHS range        [2e+00, 1e+02]
Found heuristic solution: objective 1389.0000000
Presolve removed 4 rows and 0 columns
Presolve time: 0.00s
Presolved: 22 rows, 196 columns, 557 nonzeros
Variable types: 0 continuous, 196 integer (196 binary)

Root relaxation: objective 2.571250e+03, 18 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time

     0     

**Final Selected Team**

  * 2 Goalkeepers: Pickford, Sels
  * 5 Defenders: Muñoz, Aina, Collins, Wan-Bissaka, Guéhi
  * 5 Midfielders: Salah, Mbeumo, Murphy, Iwobi, Sarr
  * 3 Forwards: Wood, Bowen, Wissa

